# PetTrace: Demo Playbook

Этот ноутбук для подготовки и показа системы: запуск, заполнение БД, проверка аналитики и ролей.

## 1. Пререквизиты

- Docker + Docker Compose
- Открытые порты: `8000`, `5432`, `6379`
- Файл окружения: `.env` (можно создать из `.env.example`)

In [ ]:
!cp -n .env.example .env
!docker compose up -d --build

## 2. Базовый bootstrap

Миграции, RBAC, справочники, системные настройки и superuser.

In [ ]:
!docker compose run --rm web python manage.py migrate --noinput
!docker compose run --rm web python manage.py bootstrap_rbac
!docker compose run --rm web python manage.py bootstrap_facilities
!docker compose run --rm web python manage.py bootstrap_system_settings
!docker compose run --rm web python manage.py create_initial_superuser

## 3. Крупное демо-заполнение

Создает большой набор владельцев, животных, приемов, лаборатории, счетов, оплат, задач и уведомлений.

In [ ]:
!docker compose run --rm web python manage.py seed_demo_data --reset --bulk-cases 240 --days 120

## 4. Демо-аккаунты

Пароль для всех пользователей ниже: `pettrace123`

- `registrar@pettrace.local`
- `vet@pettrace.local`
- `assistant@pettrace.local`
- `lab@pettrace.local`
- `inventory@pettrace.local`
- `cashier@pettrace.local`

In [ ]:
!docker compose run --rm web python manage.py shell -c 'from django.contrib.auth import get_user_model; User=get_user_model(); print("\n".join(User.objects.filter(email__in=["registrar@pettrace.local","vet@pettrace.local","assistant@pettrace.local","lab@pettrace.local","inventory@pettrace.local","cashier@pettrace.local"]).order_by("email").values_list("email", flat=True)))'

## 5. Контрольные метрики для демонстрации

Быстрая проверка, что в системе есть закрытые/завершенные приемы и данные для аналитики.

In [ ]:
!docker compose run --rm web python manage.py shell -c 'from apps.owners.models import Owner; from apps.pets.models import Pet; from apps.visits.models import Visit, Appointment; from apps.labs.models import LabOrder; from apps.billing.models import Invoice, Payment; from apps.tasks.models import Task; print("owners", Owner.objects.count()); print("pets", Pet.objects.count()); print("visits_total", Visit.objects.count()); print("visits_closed", Visit.objects.filter(status=Visit.VisitStatus.CLOSED).count()); print("visits_completed", Visit.objects.filter(status=Visit.VisitStatus.COMPLETED).count()); print("visits_in_progress", Visit.objects.filter(status=Visit.VisitStatus.IN_PROGRESS).count()); print("appointments_completed", Appointment.objects.filter(status=Appointment.AppointmentStatus.COMPLETED).count()); print("lab_done", LabOrder.objects.filter(status=LabOrder.LabOrderStatus.DONE).count()); print("invoices_paid", Invoice.objects.filter(status=Invoice.InvoiceStatus.PAID).count()); print("payments_total", Payment.objects.count()); print("tasks_total", Task.objects.count())'

## 6. Программы (модули) системы и описание

- `users`: пользователи, группы, RBAC, MFA, scope профили.
- `frontend`: web-интерфейс (дашборд, кабинеты ролей, рабочие доски).
- `facilities`: филиалы, кабинеты, оборудование, услуги.
- `owners`, `pets`, `crm`: клиентская база, питомцы, сегментация и коммуникации.
- `visits`: приемы, визиты, стационар, назначения, процедуры.
- `clinical`: протоколы, противопоказания, клинические alert/checklist.
- `labs`: лабораторные заказы, образцы, тесты, результаты.
- `inventory`: склад, партии, списания, движения остатков.
- `billing`: прайс, счета, линии, оплаты, корректировки.
- `documents`: клинические документы и генерация PDF.
- `tasks`: внутренние задачи и уведомления.
- `reports`: операционная и финансовая аналитика.
- `audit`: аудит действий.

## 7. Быстрые ссылки для показа

- UI: http://localhost:8000/
- API root: http://localhost:8000/api/
- Swagger: http://localhost:8000/api/docs/swagger/
- ReDoc: http://localhost:8000/api/docs/redoc/
- Admin: http://localhost:8000/admin/

## 8. QA перед демонстрацией

In [ ]:
!docker compose run --rm web ruff check .
!docker compose run --rm web python manage.py test --noinput

## 9. Остановка стенда

In [ ]:
!docker compose down